In [0]:
from pyspark.sql import functions as F

sales_df = spark.table("workspace.default.retail_sales_ml_apl")
inventory_df = spark.table("workspace.default.retail_inventory_ml_apl")

print("Sales:", sales_df.count())
print("Inventory:", inventory_df.count())

Sales: 125751
Inventory: 284755


In [0]:
sales_silver = sales_df.select(
    F.col("Transaction Date").alias("transaction_date"),
    F.col("Sales Type").alias("sales_type"),
    F.col("Is Return").alias("is_return"),
    F.col("Reason of Return").alias("return_reason"),
    F.col("Supplier").alias("supplier"),
    F.col("Product No").alias("product_id"),
    F.col("Product Description").alias("product_description"),
    F.col("Product Division").alias("product_division"),
    F.col("Product Category").alias("product_category"),
    F.col("Product Subcategory").alias("product_subcategory"),
    F.col("Product Segment").alias("product_segment"),
    F.col("Store").alias("store"),
    F.col("Sales Channel").alias("sales_channel"),
    F.col("Qty Sold").alias("qty_sold"),
    F.col("Sales Amount").alias("sales_amount"),
    F.col("Cogs").alias("cogs"),
    F.col("Number of Transactions").alias("transaction_count")
)

inventory_silver = inventory_df.select(
    F.col("Start Date").alias("start_date"),
    F.col("End Date").alias("end_date"),
    F.col("Stock Status").alias("stock_status"),
    F.col("Supplier").alias("supplier"),
    F.col("Product No").alias("product_id"),
    F.col("Product Description").alias("product_description"),
    F.col("Product Division").alias("product_division"),
    F.col("Product Category").alias("product_category"),
    F.col("Product Subcategory").alias("product_subcategory"),
    F.col("Product Segment").alias("product_segment"),
    F.col("Store").alias("store"),
    F.col("Store Type").alias("store_type"),
    F.col("Sales Channel").alias("sales_channel"),
    F.col("Qty on hand").alias("qty_on_hand"),
    F.col("Stocks Selling Amount").alias("stock_selling_amount"),
    F.col("Cost of Stocks").alias("cost_of_stocks"),
    F.col("Stock Unit Selling Price").alias("unit_selling_price"),
    F.col("Stock Unit Cost Price").alias("unit_cost_price")
)

In [0]:
inventory_silver = inventory_silver.withColumn(
    "is_current",
    F.when(
        F.col("end_date") == F.to_date(F.lit("9999-12-31")),
        F.lit(True)
    ).otherwise(F.lit(False))
)

In [0]:
inventory_silver = inventory_silver.withColumn(
    "end_date",
    F.when(
        F.col("end_date") == F.to_date(F.lit("9999-12-31")),
        F.lit(None).cast("date")
    ).otherwise(F.col("end_date"))
)

In [0]:
sales_silver = sales_silver.withColumn(
    "net_qty_sold",
    F.when(
        F.col("is_return") == 1,
        -F.col("qty_sold")
    ).otherwise(F.col("qty_sold"))
)

In [0]:
sales_silver = sales_silver.withColumn(
    "gross_margin",
    F.col("sales_amount") - F.col("cogs")
)

In [0]:
sales_silver = sales_silver.withColumn(
    "margin_pct",
    F.when(
        F.col("sales_amount") != 0,
        F.col("gross_margin") / F.col("sales_amount")
    ).otherwise(0)
)

In [0]:
display(
    sales_silver.select(
        "transaction_date",
        "product_id",
        "store",
        "sales_type",
        "qty_sold",
        "net_qty_sold",
        "sales_amount",
        "cogs",
        "gross_margin",
        "margin_pct"
    ).limit(20)
)

transaction_date,product_id,store,sales_type,qty_sold,net_qty_sold,sales_amount,cogs,gross_margin,margin_pct
2025-09-01,PROD-139867,STR-1006,Full Price,1.0,1.0,115.0,52.31,62.69,0.5451304347826087
2025-09-07,PROD-100449,STR-1198,Full Price,1.0,1.0,115.0,58.47,56.53,0.4915652173913044
2025-08-06,PROD-153623,STR-1157,Promo,1.0,1.0,79.99,43.76,36.23,0.4529316164520565
2025-08-31,PROD-140876,STR-1234,Full Price,1.0,1.0,110.0,50.6,59.4,0.54
2026-03-22,PROD-166113,STR-1157,Full Price,1.0,1.0,160.0,77.95,82.05,0.5128125
2025-08-27,PROD-123292,STR-1075,Full Price,1.0,1.0,115.0,51.57,63.43,0.5515652173913044
2025-11-15,PROD-149689,STR-1338,Full Price,1.0,1.0,80.0,37.6,42.4,0.53
2025-07-27,PROD-152066,STR-1253,Promo,1.0,1.0,89.99,60.06,29.929999999999993,0.33259251027891984
2025-10-25,PROD-138975,STR-1148,Full Price,1.0,1.0,31.5,20.69,10.809999999999999,0.34317460317460313
2025-07-03,PROD-158681,STR-1044,Full Price,1.0,1.0,90.0,45.04,44.96,0.4995555555555556


In [0]:
# Validate return handling

display(
    sales_silver
    .filter(F.col("is_return") == 1)
    .select(
        "transaction_date",
        "product_id",
        "store",
        "qty_sold",
        "is_return",
        "net_qty_sold",
        "sales_amount",
        "gross_margin"
    )
    .limit(20)
)

transaction_date,product_id,store,qty_sold,is_return,net_qty_sold,sales_amount,gross_margin
2026-03-14,PROD-183451,STR-1109,-1.0,1,1.0,-165.0,-81.51
2025-07-30,PROD-144341,STR-1369,-1.0,1,1.0,-69.99,-19.939999999999998
2026-03-14,PROD-188440,STR-1010,-1.0,1,1.0,-60.0,-36.6
2025-12-29,PROD-152224,STR-1006,-1.0,1,1.0,-63.0,-24.33
2025-08-23,PROD-160814,STR-1105,-1.0,1,1.0,-120.0,-59.94
2025-08-28,PROD-143184,STR-1338,-1.0,1,1.0,-110.0,-59.95
2026-01-02,PROD-100813,STR-1348,-1.0,1,1.0,-90.0,-44.49
2025-12-24,PROD-100043,STR-1074,-1.0,1,1.0,-112.0,-44.010000000000005
2025-09-14,PROD-167815,STR-1344,-1.0,1,1.0,-110.0,-54.34
2025-12-10,PROD-176331,STR-1305,-1.0,1,1.0,-81.0,-35.46


In [0]:
print("Total returns:", sales_silver.filter(F.col("is_return") == 1).count())

print(
    "Returns with positive net quantity:",
    sales_silver
    .filter(
        (F.col("is_return") == 1) &
        (F.col("net_qty_sold") > 0)
    )
    .count()
)

Total returns: 7491
Returns with positive net quantity: 7491


In [0]:
sales_silver = sales_silver.withColumn(
    "net_qty_sold",
    F.col("qty_sold")
)

print(
    "Returns with positive net quantity:",
    sales_silver
    .filter(
        (F.col("is_return") == 1) &
        (F.col("net_qty_sold") > 0)
    )
    .count()
)

Returns with positive net quantity: 0


In [0]:
display(
    sales_silver
    .filter(F.col("is_return") == 1)
    .select(
        "transaction_date",
        "product_id",
        "store",
        "qty_sold",
        "is_return",
        "net_qty_sold",
        "sales_amount",
        "gross_margin"
    )
    .limit(10)
)

transaction_date,product_id,store,qty_sold,is_return,net_qty_sold,sales_amount,gross_margin
2026-03-14,PROD-183451,STR-1109,-1.0,1,-1.0,-165.0,-81.51
2025-07-30,PROD-144341,STR-1369,-1.0,1,-1.0,-69.99,-19.939999999999998
2026-03-14,PROD-188440,STR-1010,-1.0,1,-1.0,-60.0,-36.6
2025-12-29,PROD-152224,STR-1006,-1.0,1,-1.0,-63.0,-24.33
2025-08-23,PROD-160814,STR-1105,-1.0,1,-1.0,-120.0,-59.94
2025-08-28,PROD-143184,STR-1338,-1.0,1,-1.0,-110.0,-59.95
2026-01-02,PROD-100813,STR-1348,-1.0,1,-1.0,-90.0,-44.49
2025-12-24,PROD-100043,STR-1074,-1.0,1,-1.0,-112.0,-44.010000000000005
2025-09-14,PROD-167815,STR-1344,-1.0,1,-1.0,-110.0,-54.34
2025-12-10,PROD-176331,STR-1305,-1.0,1,-1.0,-81.0,-35.46


In [0]:
# Save Silver Sales table

sales_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.silver_sales")

print("Silver Sales table created successfully.")

Silver Sales table created successfully.


In [0]:
# Save Silver Inventory table

inventory_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.silver_inventory")

print("Silver Inventory table created successfully.")

Silver Inventory table created successfully.


In [0]:
print("Silver Sales:", spark.table("workspace.default.silver_sales").count())
print("Silver Inventory:", spark.table("workspace.default.silver_inventory").count())

Silver Sales: 125751
Silver Inventory: 284755


In [0]:
inventory_silver.groupBy(
    "product_id",
    "store"
).agg(
    F.count("*").alias("inventory_records"),
    F.min("start_date").alias("first_start_date"),
    F.max("end_date").alias("last_end_date"),
    F.sum(F.when(F.col("is_current") == True, 1).otherwise(0)).alias("current_records")
).orderBy(
    F.desc("inventory_records")
).show(20, False)

+-----------+--------+-----------------+----------------+-------------+---------------+
|product_id |store   |inventory_records|first_start_date|last_end_date|current_records|
+-----------+--------+-----------------+----------------+-------------+---------------+
|PROD-100805|STR-1006|86               |2025-07-19      |2026-04-16   |1              |
|PROD-140871|STR-1080|76               |2025-06-12      |2026-04-17   |1              |
|PROD-140871|STR-1235|71               |2025-06-12      |2026-03-17   |1              |
|PROD-100805|STR-1026|70               |2025-07-19      |2026-04-16   |1              |
|PROD-100804|STR-1235|70               |2025-07-19      |2026-04-23   |1              |
|PROD-100803|STR-1006|68               |2025-07-19      |2026-04-03   |0              |
|PROD-140871|STR-1253|68               |2025-06-13      |2026-04-10   |1              |
|PROD-100805|STR-1044|67               |2025-07-21      |2026-04-04   |0              |
|PROD-100804|STR-1094|66        

In [0]:
inventory_silver.groupBy(
    "product_id",
    "store",
    "start_date"
).count().filter(
    F.col("count") > 1
).show(20, False)

+----------+-----+----------+-----+
|product_id|store|start_date|count|
+----------+-----+----------+-----+
+----------+-----+----------+-----+

